# Clusterização ST-DBSCAN

Clusterização das mudanças no ponto de origem, usando uma implementação própria do algoritmo ST-DBSCAN

In [1]:
import pandas as pd
import geopandas as gpd
import h3
import alphashape

from shapely import union_all
from shapely.geometry import MultiPoint, Point, MultiLineString
from datetime import datetime, timedelta
from pyarrow.parquet import ParquetFile
import pyarrow as pa

#from sktime.clustering.spatio_temporal import STDBSCAN
from stdbscan_kdtree import STDBSCAN

In [2]:
df_desloc = pd.read_parquet('deslocamentos.parquet')

In [3]:
cols = ['id_mudanca', 'CO_FAMILIAR_FAM',
        'DT_ATUALIZACAO_FAM', 'DT_ATUALIZACAO_FAM_pos', 'data_media',
        'LATITUDE', 'LONGITUDE', 'LATITUDE_pos', 'LONGITUDE_pos']

gdf = gpd.GeoDataFrame(df_desloc[cols], geometry=gpd.points_from_xy(df_desloc['LONGITUDE'], df_desloc['LATITUDE']), crs='EPSG:4326').to_crs('EPSG:31983')

gdf['geometry_pos'] = gpd.points_from_xy(df_desloc['LONGITUDE_pos'], df_desloc['LATITUDE_pos'], crs='EPSG:4326').to_crs('EPSG:31983')
gdf['distancia'] = gdf.apply(lambda x: round(x['geometry'].distance(x['geometry_pos']),0), axis=1)
gdf['linha_deslocamento'] = gdf['geometry'].shortest_line(gdf['geometry_pos'])

### Clusterização na origem

In [4]:
segundos_por_mes = 30 * 24 * 60 * 60

for dist, meses in [(30, 6), (60, 6), (30, 12), (60, 12), (100, 6), (100, 12)]:
    print(f"Clusterizando com eps1={dist} metros e eps2={meses} meses")
    eps_temporal_seconds = meses * segundos_por_mes
    st_dbscan = STDBSCAN(
            eps1=dist,
            eps2=eps_temporal_seconds,
            min_samples=10,
            time_col="data_media"
        )

    # Executando a clusterização
    clusters = st_dbscan.fit_predict(gdf)
    gdf[f'cluster_{dist}m_{meses}M'] = clusters

gdf.to_parquet('clusters/origem_deslocamentos_clusterizados.parquet', engine='pyarrow')

Clusterizando com eps1=30 metros e eps2=6 meses
Clusterizando com eps1=60 metros e eps2=6 meses
Clusterizando com eps1=30 metros e eps2=12 meses
Clusterizando com eps1=60 metros e eps2=12 meses
Clusterizando com eps1=100 metros e eps2=6 meses
Clusterizando com eps1=100 metros e eps2=12 meses


### Criação de polígonos de cluster na origem

In [5]:
def criar_alpha_shape(lista_pontos, alpha=0.3):
    lista_pontos = [(p.x, p.y) for p in lista_pontos]
    geometria_alpha = alphashape.alphashape(lista_pontos, alpha)
    return geometria_alpha

def criar_buffer(lista_pontos, distancia=30):
    buffers = [i.buffer(distancia) for i in lista_pontos]
    buffer = union_all(buffers)
    return buffer

def calcular_data_media(lista_datas: list[datetime]) -> datetime:
    timestamps = [dt.timestamp() for dt in lista_datas]
    media_timestamp = sum(timestamps) / len(timestamps)
    return datetime.fromtimestamp(media_timestamp)

In [6]:
# Criação de polígonos dos clusters
for dist, meses in [(30, 6), (60, 6), (30, 12), (60, 12), (100, 6), (100, 12)]:
    print(f"Agrupando com eps1={dist} metros e eps2={meses} meses")

    alpha = gdf.pivot_table(index=f'cluster_{dist}m_{meses}M', 
                            values=['CO_FAMILIAR_FAM', 'geometry', 'data_media', 'distancia'], 
                            aggfunc={'CO_FAMILIAR_FAM': 'count', 'geometry': criar_buffer, 'distancia':['median', 'mean'], 'data_media': ['min', 'max', calcular_data_media]})

    alpha.columns = ['_'.join(i) for i in alpha.columns]
    alpha = alpha.reset_index()
    alpha = (alpha.query(f'cluster_{dist}m_{meses}M != -1 & CO_FAMILIAR_FAM_count >= 10')
                .sort_values(by='CO_FAMILIAR_FAM_count', ascending=False)
                .rename(columns={'CO_FAMILIAR_FAM_count': 'num_familias', 'geometry_criar_buffer': 'geometry', 'data_media_min': 'data_inicio', 'data_media_max': 'data_fim', 'data_media_calcular_data_media': 'data_media'}))

    alpha = gpd.GeoDataFrame(alpha, geometry='geometry', crs='EPSG:31983')
    alpha.to_file(f'clusters/origem_cluster_{dist}m_{meses}M.gpkg', driver='GPKG')

    spider = gdf.query(f'cluster_{dist}m_{meses}M != -1')
    spider = spider[[f'cluster_{dist}m_{meses}M','CO_FAMILIAR_FAM', 'linha_deslocamento']]
    spider = gpd.GeoDataFrame(spider, geometry='linha_deslocamento', crs='EPSG:31983')
    spider.to_file(f'clusters/fluxos/origem_fluxo_cluster_{dist}m_{meses}M.gpkg', driver='GPKG')

Agrupando com eps1=30 metros e eps2=6 meses


c:\Users\Pedro\anaconda3\envs\geo\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()
c:\Users\Pedro\anaconda3\envs\geo\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Cannot find tms_NZTM2000.json (GDAL_DATA is not defined)
  ogr_write(


Agrupando com eps1=60 metros e eps2=6 meses
Agrupando com eps1=30 metros e eps2=12 meses
Agrupando com eps1=60 metros e eps2=12 meses
Agrupando com eps1=100 metros e eps2=6 meses
Agrupando com eps1=100 metros e eps2=12 meses
